# Dense optical flow: Farnebäck, DIS and TV-L1

Compare three library-backed dense-flow methods on synthetic translated imagery. Flow is measured in pixel units, from the first frame to the second.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from dense_optical_flow import farneback, dis

In [ ]:
rng = np.random.default_rng(22)
first = (rng.random((128, 128)) * 255).astype(np.uint8)
first = cv2.GaussianBlur(first, (5, 5), 0)
transform = np.float32([[1, 0, 4], [0, 1, -3]])
second = cv2.warpAffine(first, transform, (128, 128), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1); plt.imshow(first, cmap='gray'); plt.title('First frame')
plt.subplot(1, 2, 2); plt.imshow(second, cmap='gray'); plt.title('Translated frame'); plt.show()
print('Expected displacement: u=+4 px, v=-3 px')

In [ ]:
fb = farneback(first, second)
dis_result = dis(first, second, preset='medium')
region = np.s_[20:-20, 20:-20]
for name, result in [('Farneback', fb), ('DIS', dis_result)]:
    print(name, 'median u:', float(np.median(result['u'][region])), 'median v:', float(np.median(result['v'][region])))

## TV-L1 variational optical flow

TV-L1 uses a variational formulation with data attachment and regularisation parameters. It is a reference-library wrapper, not a from-scratch implementation.

In [ ]:
from dense_optical_flow import tvl1
tvl1_result = tvl1(first, second, num_warp=5, num_iter=20)
print('TV-L1 median u:', float(np.median(tvl1_result['u'][20:-20,20:-20])))
print('TV-L1 median v:', float(np.median(tvl1_result['v'][20:-20,20:-20])))
plt.figure(figsize=(8,3))
plt.subplot(1,2,1); plt.imshow(tvl1_result['u'], cmap='coolwarm'); plt.title('TV-L1 u (columns)')
plt.subplot(1,2,2); plt.imshow(tvl1_result['v'], cmap='coolwarm'); plt.title('TV-L1 v (rows)'); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, data, title in [(axes[0,0], fb['u'], 'Farneback u'), (axes[0,1], fb['v'], 'Farneback v'), (axes[1,0], dis_result['u'], 'DIS u'), (axes[1,1], dis_result['v'], 'DIS v')]:
    im = ax.imshow(data, cmap='coolwarm'); ax.set_title(title); fig.colorbar(im, ax=ax, shrink=.8)
plt.tight_layout(); plt.show()

## Interpretation

All three methods estimate apparent image displacement. They do not automatically measure physical ground motion. Textureless regions, occlusion, radiometric changes, residual image registration error, platform motion and terrain parallax can all produce misleading flow. Convert pixel displacement to map displacement with the raster affine transform, and evaluate on independent known-motion or tie-point data before operational use.

The wrappers use OpenCV and scikit-image; record library versions and parameters. The translation example is a smoke fixture, not a full accuracy benchmark.